# Notebook 4 — PCA Dimensionality Reduction

**Technique:** Principal Component Analysis (PCA) on 12-dimensional per-vehicle feature vectors

**Goal:** Reduce the 12-dimensional vehicle behaviour space to 2D for visualisation and pattern discovery.

## Theory

PCA finds the orthogonal directions (principal components) of maximum variance in the data.
Projecting onto PC1 and PC2 gives the 2D representation that preserves the most information.

**Why standardise first?** PCA is variance-sensitive. Without StandardScaler, features with
large numeric ranges (e.g. `totalDistanceKm` ≈ 0–100) dominate over binary-range features
(e.g. `ignitionOnFraction` ∈ [0,1]). Standardisation puts all features on equal footing.

**Cyclic hour encoding:** `hourOfDay` is not used raw. Instead we use `(sin(2πh/24), cos(2πh/24))` —
this places midnight adjacent to 11pm in 2D space, which is the correct topology for time-of-day.

In [ ]:
import boto3, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
BUCKET = 'bus-history'

FEATURE_COLS = [
    'eventCount', 'avgSpeed', 'maxSpeed', 'stdSpeed', 'idleFraction',
    'totalDistanceKm', 'avgSamplingIntervalS', 'samplingIrregularity',
    'hourSin', 'hourCos', 'headingChanges', 'ignitionOnFraction'
]

In [ ]:
def load_feature_vectors(max_objects=5000):
    records = []
    paginator = s3.get_paginator('list_objects_v2')
    count = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix='features/vehicle-hourly/'):
        for obj in page.get('Contents', []):
            if count >= max_objects: break
            body = s3.get_object(Bucket=BUCKET, Key=obj['Key'])['Body'].read()
            records.append(json.loads(body))
            count += 1
    return pd.DataFrame(records)

df = load_feature_vectors()
print(f'Loaded {len(df):,} feature vectors')
print(df[['routeNo','hourOfDay','avgSpeed','eventCount']].head())

In [ ]:
# ── Prepare matrix ────────────────────────────────────────────────────────
for col in FEATURE_COLS:
    df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0)

X = df[FEATURE_COLS].values
X_scaled = StandardScaler().fit_transform(X)

print(f'Feature matrix shape: {X_scaled.shape}')
print('Mean after scaling:', X_scaled.mean(axis=0).round(3))
print('Std  after scaling:', X_scaled.std(axis=0).round(3))

In [ ]:
# ── Fit PCA ────────────────────────────────────────────────────────────────
pca = PCA(n_components=min(12, len(df)))
X_pca = pca.fit_transform(X_scaled)

print('Explained variance ratio per component:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.3f} ({100*var:.1f}%)')
print(f'\nPC1+PC2 captures {100*pca.explained_variance_ratio_[:2].sum():.1f}% of total variance')

In [ ]:
# ── Scree plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_, color='#4a90d9')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cum_var)+1), cum_var, marker='o', color='#e94560')
axes[1].axhline(0.9, linestyle='--', color='gray', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance')
axes[1].legend()
plt.tight_layout()
plt.savefig('/tmp/pca_scree.png', dpi=150)
plt.show()

In [ ]:
# ── 2D scatter: colour by route ───────────────────────────────────────────
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]

top_routes = df['routeNo'].value_counts().head(10).index.tolist()
plot_df = df[df['routeNo'].isin(top_routes)]

fig, ax = plt.subplots(figsize=(10, 7))
for route in top_routes:
    subset = plot_df[plot_df['routeNo'] == route]
    ax.scatter(subset['PC1'], subset['PC2'], label=route, alpha=0.5, s=20)

ax.set_xlabel(f'PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% variance)')
ax.set_title('PCA 2D Projection — Vehicle Hourly Behaviour by Route')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('/tmp/pca_scatter.png', dpi=150)
plt.show()

In [ ]:
# ── Component loadings heatmap ────────────────────────────────────────────
# Shows how much each original feature contributes to each PC
loadings = pd.DataFrame(
    pca.components_[:4].T,
    index=FEATURE_COLS,
    columns=[f'PC{i+1}' for i in range(4)]
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('PCA Loadings — Feature Contributions to PC1–PC4')
plt.tight_layout()
plt.savefig('/tmp/pca_loadings.png', dpi=150)
plt.show()
print('Interpretation: PC1 dominated by speed features = activity axis')
print('               PC2 dominated by sampling features = data quality axis')